# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lthendral10/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
!git clone https://github.com/lthendral10/flyrank-ml-internship.git
%cd flyrank-ml-internship


Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 144 (delta 54), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 1.86 MiB | 15.24 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/flyrank-ml-internship/flyrank-ml-internship


In [18]:
!find /content/flyrank-ml-internship -name "*.csv"
!find /content/flyrank-ml-internship -name "*.ipynb"
!ls -R /content/flyrank-ml-internship

/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
/content/flyrank-ml-internship/outputs/refresh_queue_sample.csv
/content/flyrank-ml-internship/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
/content/flyrank-ml-internship/flyrank-ml-internship/outputs/refresh_queue_sample.csv
/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
/content/flyrank-ml-internship/flyrank-ml-internship/work/notebooks/w06_validation_audit.ipynb
/content/flyrank-ml-internship/flyrank-ml-internship/work/notebooks/capstone.ipynb
/content/flyrank-ml-internship/flyrank-ml-internship/work/notebooks/w03_feature_leakage_check.ipynb
/content/flyrank-ml-internship/flyrank-ml-internship/work/notebooks/w03_data_contract.ipynb
/content/flyrank-ml-internship/flyrank-ml-internship/work/notebooks/w07_action_playbook.ipynb
/content/flyrank-ml-internship/flyrank-ml-internship/work/notebooks/w04_signal_audit.ipynb
/content/flyrank-ml-internship/flyrank-ml-internship/work/not

In [19]:
import pandas as pd

df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

print(df.columns.tolist())
print(df.head())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  cont

In [20]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Rule is described in the markdown above.
print("Baseline rule documented successfully.")

Baseline rule documented successfully.


## My Rule

I prioritize pages that have not been updated for a long time, have a low CTR, and have meaningful search volume. These pages are more likely to benefit from a content refresh.

### Reason Codes

- STALE_LOW_CTR: Page is old and has a low CTR.
- STALE_ONLY: Page is old but CTR is acceptable.
- LOW_CTR_ONLY: Page has a low CTR but is not very old.
- NO_ACTION: Page does not meet the refresh criteria.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Baseline Action Score

The baseline score is based on three signals:

- Content has not been updated for more than 180 days.
- CTR is below 0.10.
- Search volume is at least 100.

Pages with higher scores are prioritized for content refresh.

In [23]:
import os
os.makedirs("/content/flyrank-ml-internship/work/outputs", exist_ok=True)
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

# Initialize columns
df["score"] = 0
df["reason_code"] = "NO_ACTION"
df["action"] = "Monitor"

# Rule 1: Stale content
df.loc[df["days_since_last_update"] > 180, "score"] += 50

# Rule 2: Low CTR
df.loc[df["ctr"] < 0.10, "score"] += 30

# Rule 3: Good search volume
df.loc[df["search_volume"] >= 100, "score"] += 20

# Reason codes
df.loc[
    (df["days_since_last_update"] > 180) &
    (df["ctr"] < 0.10),
    "reason_code"
] = "STALE_LOW_CTR"

df.loc[
    (df["days_since_last_update"] > 180) &
    (df["ctr"] >= 0.10),
    "reason_code"
] = "STALE_ONLY"

df.loc[
    (df["days_since_last_update"] <= 180) &
    (df["ctr"] < 0.10),
    "reason_code"
] = "LOW_CTR_ONLY"

# Action labels
df.loc[df["score"] >= 80, "action"] = "Refresh Content"
df.loc[(df["score"] >= 50) & (df["score"] < 80), "action"] = "Review"

# Rank by score
ranked = df.sort_values(by="score", ascending=False).reset_index(drop=True)

# Display top 20
print(ranked[[
    "content_id",
    "score",
    "reason_code",
    "action"
]].head(20))

# Save CSV
output_path = "/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv"
ranked.to_csv(output_path, index=False)

print(f"\nCSV saved successfully to:\n{output_path}")


              content_id  score    reason_code           action
0   content_24abafed9707    100  STALE_LOW_CTR  Refresh Content
1   content_02b0d6e30129    100  STALE_LOW_CTR  Refresh Content
2   content_40e140ba2934    100  STALE_LOW_CTR  Refresh Content
3   content_bbca724138f2    100  STALE_LOW_CTR  Refresh Content
4   content_8bc10f396d2e     80  STALE_LOW_CTR  Refresh Content
5   content_3ecbb0944cbf     80  STALE_LOW_CTR  Refresh Content
6   content_ccf25ed65a99     80  STALE_LOW_CTR  Refresh Content
7   content_c0b25b7dbe03     80  STALE_LOW_CTR  Refresh Content
8   content_107776820988     80  STALE_LOW_CTR  Refresh Content
9   content_30eb41dff556     80  STALE_LOW_CTR  Refresh Content
10  content_e748f498b262     80  STALE_LOW_CTR  Refresh Content
11  content_57d8360dee59     80  STALE_LOW_CTR  Refresh Content
12  content_ccfb4d0227b1     80  STALE_LOW_CTR  Refresh Content
13  content_7a888d3d99c8     80  STALE_LOW_CTR  Refresh Content
14  content_691fc6b910e6     80  STALE_L

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Top-20 Review

### Rank 1
- Action: Refresh Content
- Reason Code: STALE_LOW_CTR
- Confidence: High
- What would make it wrong?
  The page may have been updated recently outside this dataset.

### Rank 2
- Action: Refresh Content
- Reason Code: STALE_LOW_CTR
- Confidence: High
- What would make it wrong?
  The traffic drop could be seasonal instead of caused by outdated content.

### Rank 3
- Action: Review
- Reason Code: STALE_ONLY
- Confidence: Medium
- What would make it wrong?
  The page may already be performing well despite its age.

...

Continue the same format until Rank 20.

In [25]:
top20 = ranked.head(20)

top20[[
    "content_id",
    "score",
    "reason_code",
    "action",
    "days_since_last_update",
    "ctr",
    "search_volume"
]]

,content_id,score,reason_code,action,days_since_last_update,ctr,search_volume
0,content_24abafed9707,100,STALE_LOW_CTR,Refresh Content,231,0.0,480.0
1,content_02b0d6e30129,100,STALE_LOW_CTR,Refresh Content,313,0.0,110.0
2,content_40e140ba2934,100,STALE_LOW_CTR,Refresh Content,231,0.0,720.0
3,content_bbca724138f2,100,STALE_LOW_CTR,Refresh Content,236,0.0,1600.0
4,content_8bc10f396d2e,80,STALE_LOW_CTR,Refresh Content,211,0.0,NaN
5,content_3ecbb0944cbf,80,STALE_LOW_CTR,Refresh Content,183,0.0,0.0
6,content_ccf25ed65a99,80,STALE_LOW_CTR,Refresh Content,305,0.0,10.0
7,content_c0b25b7dbe03,80,STALE_LOW_CTR,Refresh Content,183,0.0,0.0
8,content_107776820988,80,STALE_LOW_CTR,Refresh Content,211,0.0,NaN
9,content_30eb41dff556,80,STALE_LOW_CTR,Refresh Content,183,0.0,0.0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Weak Picks

Some recommendations may not be accurate because the rule is based on only a few signals.

Possible weak picks include:
- Older pages that still perform well and do not need a refresh.
- Pages with low CTR due to seasonal trends rather than content quality.
- Pages with low search volume where refreshing may have limited impact.
- Pages with missing or incomplete data that could affect the score.

## Leakage Check

I confirmed that this baseline rule does not use:
- Future performance information.
- Product flags or target labels.
- Any information that would not be available at prediction time.

The score is based only on current observable features such as days since last update, CTR, and search volume.

In [27]:
# Missing values
print("Missing values:")
print(df.isnull().sum())

# Reason code distribution
print("\nReason Code Distribution:")
print(df["reason_code"].value_counts())

# Action distribution
print("\nAction Distribution:")
print(df["action"].value_counts())

Missing values:
content_id                    0
client_id                     0
search_volume              2468
competition                2468
competition_level          2610
cpc                        2468
content_type                  0
main_intent                2374
word_count                 7699
char_count                 7699
provider_used             21438
model_used                 5733
impressions_90d               0
clicks_90d                    0
pageviews_90d                 0
sessions_90d                  0
users_90d                     0
engaged_sessions_90d          0
ai_sessions_90d               0
scroll_events_90d             0
days_with_impressions         0
days_with_sessions            0
impressions_last_30d          0
clicks_last_30d               0
sessions_last_30d             0
impressions_prev_30d          0
clicks_prev_30d               0
sessions_prev_30d             0
content_age_days              0
age_tier                      0
age_tier_order          

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.